# S21 · Which dark stores to open (facility location)

A quick-commerce company (think Blinkit or Zepto) can open small neighbourhood
"dark stores" across a metro. Each store it opens costs rent, but a nearby store
means cheaper, faster delivery. We build a small model in PuLP that decides which
stores to open and who each one serves, for the cheapest overall plan.

**New here? Read this once.**

- New to Python? Press play on each cell, top to bottom, and read the plain note
  above each one.
- New to the idea of a goal, some rules, and a solver? Open
  `primers/optimization_objectives_and_constraints.md` first.
- Want the packing and matching warm-up? That is notebook
  `01_knapsack_assignment.ipynb`. This one carries straight on from it.
- Already confident? Run the simple cells, then look for the **Stretch
  (optional)** cell near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# The only library Colab does not already ship is PuLP, so we install just that.
import sys
if "google.colab" in sys.modules:
    !pip install -q "pulp<4"
else:
    print("Not on Colab - assuming the libraries are already installed.")

In [ ]:
import pulp                 # writes the model and hands it to a free solver (CBC)
import numpy as np             # fast maths on tables of numbers

## Step 1 — the problem in plain words

The company can open a dark store at a few **candidate locations**. Opening a
store costs a fixed monthly amount. Once open, a store can serve delivery zones,
and serving a zone costs more the further away it is.

There is a trade-off. Open **more** stores and you pay more rent but deliver over
shorter distances; open **fewer** and you save rent but deliver from further away.
We want the cheapest plan overall. This puzzle is called **facility location**.

In [ ]:
# Set a seed so the made-up costs are the same every run.
np.random.seed(0)

# Three candidate store locations and the fixed monthly cost of opening each.
store_names   = ["Andheri store", "Dadar store", "Thane store"]
opening_cost  = [50, 40, 45]           # fixed cost to open each store

# Four delivery zones we must serve.
zone_names = ["Zone 1", "Zone 2", "Zone 3", "Zone 4"]

number_of_stores = len(store_names)
number_of_zones = len(zone_names)

print("Candidate stores:", number_of_stores)
print("Zones to serve  :", number_of_zones)

## Step 2 — the cost of serving each zone from each store

`serve_cost[i][j]` is the cost for store `i` to serve zone `j` (think of it as
the delivery distance turned into rupees). We make up small whole numbers so the
answer is easy to check by eye.

In [ ]:
# serve_cost[i][j] = cost for store i to serve zone j.
# We use random whole numbers between 4 and 19.
serve_cost = np.random.randint(4, 20, size=(number_of_stores, number_of_zones))

print("Serving-cost table (rows = stores, columns = zones):")
print(serve_cost)

## Step 3 — create the problem

We are **minimising** total cost: opening costs plus serving costs.

In [ ]:
# LpMinimize because we want the cheapest plan.
facility = pulp.LpProblem("facility_location", pulp.LpMinimize)

print("Created the facility-location problem (minimise total cost).")

## Step 4 — the yes/no decisions

Two kinds of yes/no decision:

- `is_open[i]` = 1 if we open store `i`, else 0.
- `serves[i][j]` = 1 if store `i` serves zone `j`, else 0.

Both are binary. A model that mixed these whole-number decisions with ordinary
smooth ones would be called a **mixed-integer program**, or MIP. Here everything
is 0/1.

In [ ]:
# One open/closed decision per store.
is_open = []
for store_index in range(number_of_stores):
    variable = pulp.LpVariable("open_" + str(store_index), cat="Binary")
    is_open.append(variable)

# One serve/not-serve decision per (store, zone) pair.
serves = []
for store_index in range(number_of_stores):
    row = []
    for zone_index in range(number_of_zones):
        variable = pulp.LpVariable("serve_" + str(store_index) + "_" + str(zone_index),
                                   cat="Binary")
        row.append(variable)
    serves.append(row)

print("Created", number_of_stores, "open/closed variables and",
      number_of_stores * number_of_zones, "serve variables.")

## Step 5 — the goal: opening cost + serving cost

Add up the opening cost of every store we open, plus the serving cost of every
(store, zone) pair we switch on. We build each sum with a plain `for` loop.

In [ ]:
# Part 1: cost of opening stores.
opening_terms = []
for store_index in range(number_of_stores):
    opening_terms.append(opening_cost[store_index] * is_open[store_index])
total_opening_cost = pulp.lpSum(opening_terms)

# Part 2: cost of serving zones.
serving_terms = []
for store_index in range(number_of_stores):
    for zone_index in range(number_of_zones):
        this_cost = int(serve_cost[store_index][zone_index])
        serving_terms.append(this_cost * serves[store_index][zone_index])
total_serving_cost = pulp.lpSum(serving_terms)

# The goal is the sum of both parts.
facility += total_opening_cost + total_serving_cost

print("Goal added: opening cost + serving cost.")

## Step 6 — rule: every zone is served exactly once

For each zone, exactly one store must serve it, so the serve variables in that
zone's column must sum to 1.

In [ ]:
# Every zone must be served by exactly one store.
for zone_index in range(number_of_zones):
    column_terms = []
    for store_index in range(number_of_stores):
        column_terms.append(serves[store_index][zone_index])
    facility += pulp.lpSum(column_terms) == 1

print("Added: each zone is served by exactly one store.")

## Step 7 — the key linking rule

You may only serve a zone **from a store that is open**. Written as a rule this is
`serves[i][j] <= is_open[i]`: if store `i` serves anyone (`serves = 1`), then
`is_open[i]` is forced up to 1, so we must pay to open it. This "if-then" written
as an inequality is the most important modelling trick in integer programming.

In [ ]:
# Linking rule: cannot serve from a closed store.
for store_index in range(number_of_stores):
    for zone_index in range(number_of_zones):
        facility += serves[store_index][zone_index] <= is_open[store_index]

print("Added the linking rule: serve only from an open store.")

## Step 8 — let the solver choose the cheapest plan

In [ ]:
facility.solve(pulp.PULP_CBC_CMD(msg=0))
print("Solver status:", pulp.LpStatus[facility.status])
print("Lowest total cost found:", pulp.value(facility.objective))

## Step 9 — which stores open, and who they serve

We print the stores the solver decided to open, and for each one the zones it
serves.

In [ ]:
print("Plan chosen by the solver:")
print()
for store_index in range(number_of_stores):
    if pulp.value(is_open[store_index]) == 1:
        print(store_names[store_index], "is OPEN (opening cost",
              opening_cost[store_index], ")")
        for zone_index in range(number_of_zones):
            if pulp.value(serves[store_index][zone_index]) == 1:
                print("    serves", zone_names[zone_index],
                      "(serving cost", int(serve_cost[store_index][zone_index]), ")")
    else:
        print(store_names[store_index], "is closed")

### Stretch (optional) — watch the trade-off move

Skip this if you are new to code. Here is a nice way to feel the trade-off. If
rent were much cheaper, the solver would open **more** stores, because short
deliveries would be worth the extra rent. Let's rebuild the same model with the
opening costs cut to a fifth and see how the plan changes.

In [ ]:
# Same model, but with much cheaper opening costs.
cheap_opening_cost = []
for store_index in range(number_of_stores):
    cheap_opening_cost.append(opening_cost[store_index] // 5)

cheap = pulp.LpProblem("facility_cheap_rent", pulp.LpMinimize)

open_cheap = [pulp.LpVariable("openc_" + str(i), cat="Binary")
              for i in range(number_of_stores)]
serve_cheap = [[pulp.LpVariable("servec_" + str(i) + "_" + str(j), cat="Binary")
                for j in range(number_of_zones)] for i in range(number_of_stores)]

cheap += (pulp.lpSum(cheap_opening_cost[i] * open_cheap[i]
                     for i in range(number_of_stores))
          + pulp.lpSum(int(serve_cost[i][j]) * serve_cheap[i][j]
                       for i in range(number_of_stores)
                       for j in range(number_of_zones)))

for j in range(number_of_zones):
    cheap += pulp.lpSum(serve_cheap[i][j] for i in range(number_of_stores)) == 1
for i in range(number_of_stores):
    for j in range(number_of_zones):
        cheap += serve_cheap[i][j] <= open_cheap[i]

cheap.solve(pulp.PULP_CBC_CMD(msg=0))

number_open_before = sum(int(pulp.value(is_open[i])) for i in range(number_of_stores))
number_open_after = sum(int(pulp.value(open_cheap[i])) for i in range(number_of_stores))
print("Stores open with normal rent :", number_open_before)
print("Stores open with cheap rent  :", number_open_after)
print("Cheaper rent tends to open more, nearer stores - that is the trade-off.")

## What you just did

You built a facility-location model: yes/no decisions about **opening** stores and
**assigning** zones, tied together by the linking rule that you can only serve
from an open store. The solver balanced rent against delivery cost and returned
the cheapest plan. This is exactly the kind of question Blinkit, Zepto or Flipkart
faces when choosing where to place stores and fulfilment centres across a city.

Next notebook: `03_last_mile_routing.ipynb`, where a single van has to visit many
stops and we meet the reason real routing is so hard.